# Latest Recirculation Flow-Meter Temperature

This notebook finds the newest `data/raw/recirculation/log_*.csv` file by timestamp in the file name and plots the flow-meter temperature versus elapsed time.

In [1]:
from pathlib import Path
import re
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

warnings.filterwarnings(
    'ignore',
    message='FigureCanvasAgg is non-interactive, and thus cannot be shown',
    category=UserWarning,
)

NB_PATH = Path.cwd()
REPO_ROOT = NB_PATH
for candidate in [NB_PATH, *NB_PATH.parents]:
    if (candidate / 'data').exists() and (candidate / 'analysis').exists():
        REPO_ROOT = candidate
        break

analysis_src = REPO_ROOT / 'analysis' / 'src'
if str(analysis_src) not in sys.path:
    sys.path.insert(0, str(analysis_src))

import orca.logbook as logbook

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

TC_CALIBRATION_PATH = REPO_ROOT / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'

print(f'Repo root: {REPO_ROOT}')
print(f'TC calibration: {TC_CALIBRATION_PATH}')

Repo root: /home/aamy/Documents/hfe-system
TC calibration: /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260420.csv


In [2]:
RECIRCULATION_DIR = REPO_ROOT / 'data' / 'raw' / 'recirculation'
LOG_PATTERN = re.compile(r'log_(\d{8})_(\d{6})')


def log_timestamp(path):
    match = LOG_PATTERN.search(path.name)
    if match is None:
        return pd.Timestamp.min
    return pd.to_datetime(''.join(match.groups()), format='%Y%m%d%H%M%S')


log_files = sorted(RECIRCULATION_DIR.glob('log_*.csv'))
if not log_files:
    raise FileNotFoundError(f'No recirculation logs found in {RECIRCULATION_DIR}')

LOG_PATH = max(log_files, key=log_timestamp)
raw = logbook.read_tc_calibrated_csv(LOG_PATH, calibration_path=TC_CALIBRATION_PATH)
data, flow_note = logbook.add_canonical_flow_columns(
    raw,
    density_bounds=(1200.0, 1800.0),
    log_path=LOG_PATH,
)
data = data.sort_values('time_s').reset_index(drop=True)
data['time_s'] = pd.to_numeric(data['time_s'], errors='coerce')
data['temperature_c_si'] = pd.to_numeric(data['temperature_c_si'], errors='coerce')
data['elapsed_min'] = (data['time_s'] - float(data['time_s'].iloc[0])) / 60.0

valid = data[['elapsed_min', 'time_s', 'temperature_c_si']].dropna()
if valid.empty:
    raise ValueError(f'No valid flow-meter temperature samples found in {LOG_PATH}')

summary = pd.Series({
    'log_file': LOG_PATH.name,
    'log_timestamp': log_timestamp(LOG_PATH),
    'samples': int(len(valid)),
    'duration_min': float(valid['elapsed_min'].max() - valid['elapsed_min'].min()),
    'temperature_min_C': float(valid['temperature_c_si'].min()),
    'temperature_max_C': float(valid['temperature_c_si'].max()),
    'restored_tc_calibration_applied': bool(data.attrs.get('tc_calibration_applied_from_log_metadata', False)),
    'flow_note': flow_note,
}, name='value')

display(summary.to_frame())

,value
log_file,log_20260424_153546.csv
log_timestamp,2026-04-24 15:35:46
samples,9467
duration_min,330.042333
temperature_min_C,-94.38
temperature_max_C,21.567
restored_tc_calibration_applied,True
flow_note,"This CSV already looks SI-like, so the noteboo..."


In [3]:
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)

ax.plot(
    valid['elapsed_min'],
    valid['temperature_c_si'],
    color='#111827',
    lw=1.35,
    label='Flow-meter temperature',
)
ax.scatter(
    valid['elapsed_min'],
    valid['temperature_c_si'],
    s=7,
    color='#2563eb',
    alpha=0.28,
    linewidths=0,
)

min_row = valid.loc[valid['temperature_c_si'].idxmin()]
ax.scatter(
    [min_row['elapsed_min']],
    [min_row['temperature_c_si']],
    s=36,
    color='#dc2626',
    zorder=3,
    label='Minimum',
)
ax.annotate(
    f"min {min_row['temperature_c_si']:.1f} C",
    xy=(min_row['elapsed_min'], min_row['temperature_c_si']),
    xytext=(8, 8),
    textcoords='offset points',
    fontsize=9,
    color='#991b1b',
)

ax.set_title(f'Flow-meter temperature vs elapsed time - {LOG_PATH.name}')
ax.set_xlabel('Elapsed time from first sample [min]')
ax.set_ylabel('Flow-meter temperature [C]')
ax.legend(loc='best')
ax.minorticks_on()
ax.grid(True, which='major', alpha=0.34, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.14, linestyle=':', linewidth=0.6)

plt.show()